In [1]:
import polars as pl
import numpy as np
import datetime as dt
import os
import sys
import json 
import importlib

sys.path.append("../utils/")

import helpers as hp

In [2]:
rng = np.random.default_rng(seed=274)

In [3]:

with open("../configs/zone_distances.json", "r") as json_file:
    zone_distances = json.load(json_file)

In [4]:
zone_distances

{'Yerbabuena': {'centroids': [20.964404421308334, -101.2847459818324],
  'points': [{'point': [20.9743, -101.300626],
    'distance': 0.01871089133756122},
   {'point': [20.946278, -101.298483], 'distance': 0.022743632462390598},
   {'point': [20.951706, -101.264939], 'distance': 0.023527992541503697},
   {'point': [20.979447, -101.274624], 'distance': 0.018131014585796808},
   {'point': [20.985938, -101.285636], 'distance': 0.02155196379935847},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122}]},
 'Marfil': {'centroids': [20.998877268148007, -101.29015919031652],
  'points': [{'point': [20.979212, -101.292779],
    'distance': 0.019839006379117525},
   {'point': [20.991242, -101.304647], 'distance': 0.016376628136369357},
   {'point': [21.006043, -101.294156], 'distance': 0.008205010702043177},
   {'point': [21.019272, -101.283363], 'distance': 0.021497285645706604},
   {'point': [21.008196, -101.275124], 'distance': 0.017688858391175503},
   {'point': [20.999863,

In [5]:
gym_hours = {"open" : "6.0",
             "close" : "22.0"}

profiles = {
"1": {
"name" : "frequent_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .89},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .76},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .85},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .7},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .9},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "6.43",
"hour_mean" : ".75"
},
"2" : {
"name" : "frequent_noon",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .7},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .82},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .67},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .87},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.78},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .84},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "13.16",
"hour_mean" : "1.40"
            },
"3" :{
"name" : "frequent_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .87},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .76},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .84},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "19.5",
"hour_mean" : "1.15"
},
"4" :{
"name" : "random_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .83},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .68},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 0.86},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "8.5",
"hour_mean" : "2.05"
},
"5" :{
"name" : "random_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .8},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : 1},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .8},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "20.1",
"hour_mean" : "2.08"
},
"6" :{
"name" : "full_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .5},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .5},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.5},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .5},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.00",
"hour_mean" : "3.0"
},
"7" :{
"name" : "rare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .2},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .4},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .2},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.3},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .4},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "16.0",
"hour_mean" : "3.5"
},
"8" :{
"name" : "ultrarare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .1},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .1},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.2},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.5",
"hour_mean" : "5.5"
}
}

profile_weights = {"1" : 0.23076923, "2": 0.12820513, "3" : 0.25641026, "4" : 0.07692308, "5" : 0.07692308, "6" : 0.05128205, "7" : 0.07692308, "8" : 0.1025641}

for profile in profiles.keys():
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
        
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    
    profiles[profile]["total_weight"] = sum_weight

profiles

{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': '.75',
  'total_weight': 5.0},
 '2': {'name': 'frequent_noon',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
   

In [6]:
# Functions
def create_customer_profiles(profile_weights_ : dict, n_):
    return {str(i): str(int(rng.choice( list(profile_weights.keys()), p = list(profile_weights.values()) ) ) ) for i in range(n_) }

In [7]:
create_customer_profiles(profiles, 9)

{'0': '8',
 '1': '1',
 '2': '3',
 '3': '7',
 '4': '3',
 '5': '4',
 '6': '1',
 '7': '5',
 '8': '1'}

In [8]:
# prob = np.array((.9,0.5,1,.3,.3,.2,.3,.4))

# prob/sum(prob)

In [9]:
importlib.reload(hp)

oli =hp.create_customer(zone_distances)

Creating customer
36254


In [10]:
oli

{'id': None,
 'name': 'Rosalia Ariadna',
 'last_name': 'Salinas Zambrano',
 'gender': np.False_,
 'birth_date': datetime.datetime(1993, 9, 7, 0, 0),
 'lat': 20.98821795202615,
 'lon': -101.29917077337164,
 'zipcode': 36254,
 'email': 'g**************r@gmail.com',
 'phone_number': '(473)8821424',
 'created_at': datetime.datetime(2026, 5, 22, 11, 0, 27, 235821),
 'status': 'Active',
 'profile_type': None,
 'updated_at': datetime.datetime(2026, 5, 22, 11, 0, 27, 235827)}

In [ ]:
date = '2026-01-01'

def create_customer_access_data(date_, profile_metadata_, gym_hours_, data_path_ = "../data/", customers_file_ = "customers.csv", payment_file_ = "payments.csv", access_file_ = "access.csv"):
    
    # Define Schemas
    customers_schema = {"Customer_Id" :pl.Int64,
                        "Name": pl.String,
                        "Last_Name": pl.String,
                        "Gender" : pl.Boolean,
                        "Birth_Date" : pl.Datetime,
                        "Latitude": pl.Float32,
                        "Longitude" : pl.Float32,
                        "Zipcode" : pl.Int64 ,
                        "Email" : pl.String,
                        "Phone_Number" : pl.String,
                        "Created_At" : pl.Datetime,
                        "Status" : pl.String,
                        "Profile_Type" : pl.String,
                        "Updated_At" : pl.Datetime}
    

    payments_schema = {"Payment_Id" : pl.String,
                       "Plan_Id" : pl.String,
                       "Customer_Id" : pl.Int64,
                       "Payment_Amount" : pl.Float16,
                       "Payment_Time" : pl.Datetime,
                       "Plan_Expiration_Time" : pl.Datetime,
                       "Payment_Status": pl.String,
                       "Updated_At" : pl.Datetime
    }

    access_schema = {"Visit_Id" : pl.Int64,
                     "Customer_Id" : pl.Int64,
                     "Plan_Id" : pl.String,
                     "Branch_Id" : pl.Float16,
                     "Payment_Id" : pl.Datetime,
                     "Date" : pl.Datetime,
                     "Updated_At" : pl.Datetime
    }

    eval_date = dt.datetime.strptime(date_, "%Y-%m-%d")
    day_names_dic = {0 : 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4:'Friday', 5:'Saturday', 6: 'Sunday'}
    day_of_week = eval_date.weekday()
    day_name = day_names_dic[day_of_week]

    print(eval_date, day_of_week, day_name)
    
    if not os.path.isdir(data_path_):
        os.makedirs(data_path_, exist_ok = True)

    if not os.path.exists(data_path_ + customers_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = customers_schema,
                                                    orient="row").write_csv(data_path_+ customers_file_)
    
    if not os.path.exists(data_path_ + payment_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = payments_schema,
                                                    orient="row").write_csv(data_path_+ payment_file_)
        
    if not os.path.exists(data_path_ + access_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = access_schema,
                                                    orient="row").write_csv(data_path_+ access_file_)

    df_customers = pl.read_csv(data_path_+ customers_file_, schema=customers_schema)

    #########################
    # 1. Create new customers
    #########################

    # 1.1. Customers configs
    tot_new = int(abs(rng.normal(0, 1)))
    print("Total new customers", tot_new)

    max_id = df_customers.select(pl.max("Id_Customer")).item()

    counter = max_id + 1
    if max_id == None:
        counter = 1

    dict_profiles = create_customer_profiles(profile_metadata_, tot_new)

    # 1.2 Generate customers
    new_customers = []
    for new_cus in range(tot_new):

        new_customer = hp.create_customer(zone_distances)
        new_customer["id"] = counter
        new_customer["profile_type"] = dict_profiles[str(new_cus)]
        new_customer["created_at"] = dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

        new_customers.append(tuple(new_customer.values()))
        counter +=1

    # 1.3 Save file
    df_new_customers = pl.DataFrame(new_customers, customers_schema, orient = "row")
    df_customers = pl.concat([df_customers, df_new_customers])
    df_customers.write_csv(data_path_ + customers_file_)

    ###########################
    # 2. Create Access Registry
    ###########################

    # 2.1 Given at least one customer, generate random visits
    if df_customers.shape[0] >0:
        temp = df_customers.filter(pl.col("Status") =="Active").to_dicts()

        for row in temp:
            profile = str(row["Profile_Type"])
            profile_data = profile_metadata_[profile]
            day_proba  = profile_data["days_probability"][day_name[0:3].lower()]["day_probability"]

            has_visit = rng.choice([1,0], p = [day_proba, 1-day_proba])
            print(profile, day_proba, has_visit)

            # Check is customer assisted
            if has_visit == 0:
                continue
            
            
            


    
    
    
    
    
    
    
    return df_customers
    

create_customer_access_data(date, profiles, gym_hours)

2026-01-02 00:00:00 4 Friday
Total new customers 1
Creating customer
36250
8 0.2 1
3 0.09070294784580499 0
1 0.18 1
3 0.09070294784580499 0
1 0.18 0
5 0.08333333333333334 0
7 0.1764705882352941 0


Id_Customer,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,datetime[μs],f32,f32,i64,str,str,datetime[μs],str,str,datetime[μs]
1,"""Benito Gabino""","""Maldonado Otero""",true,2004-08-20 00:00:00,21.017767,-101.254303,36000,"""a**********e@hotmail.com""","""(473)7984438""",2026-01-01 20:47:13.525995,"""Active""","""8""",2026-05-20 20:47:13.525957
1,"""Salvador Salvador""","""Corral Covarrubias""",true,1989-12-18 00:00:00,21.024862,-101.25441,36013,"""n***************h@hotmail.com""","""(473)9898681""",2026-01-01 11:00:30.986310,"""Active""","""3""",2026-05-22 11:00:30.986267
2,"""Gerardo Claudio""","""Perea Montemayor""",true,2006-02-21 00:00:00,21.00334,-101.288254,36253,"""c****************y@hotmail.com""","""(473)5660807""",2026-01-01 11:00:34.684226,"""Active""","""1""",2026-05-22 11:00:34.684184
2,"""Susana Ivonne""","""Posada Barela""",false,2004-07-20 00:00:00,21.021021,-101.262688,36030,"""l***************u@gmail.com""","""(473)8059431""",2026-01-01 11:04:48.938977,"""Active""","""3""",2026-05-22 11:04:48.938938
3,"""Fidel Carlos""","""Yáñez Saldaña""",true,2009-12-02 00:00:00,21.014194,-101.265236,36086,"""w********i@gmail.com""","""(473)5287675""",2026-01-01 11:04:52.644673,"""Active""","""1""",2026-05-22 11:04:52.644633
4,"""Gustavo Ramiro""","""Ybarra Polanco""",true,1994-06-24 00:00:00,20.975775,-101.27916,36256,"""w***************o@gmail.com""","""(473)6764659""",2026-01-01 11:05:28.875461,"""Active""","""5""",2026-05-22 11:05:28.875422
5,"""Citlali Lucía""","""Quintero Luna""",false,1978-05-21 00:00:00,21.007978,-101.274376,36250,"""a***************r@gmail.com""","""(473)9855400""",2026-01-02 11:06:56.202090,"""Active""","""7""",2026-05-22 11:06:56.202050


In [12]:
for profile in profiles.keys():
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
    
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    profiles[profile]["total_weight"] = sum_weight
profiles
        

{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': '.75',
  'total_weight': 5.0},
 '2': {'name': 'frequent_noon',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
   

In [13]:
dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

datetime.datetime(2026, 1, 1, 11, 0, 34, 763406)

In [14]:
my_dic = {}


for i in range(100):
    value = int(abs(rng.normal(0, 1.5)))
    my_dic[value] = my_dic.get(value, 0) + 1

my_dic

{2: 8, 0: 52, 1: 36, 3: 3, 4: 1}